# Qwen on P5 (text purpose): standalone detection + system check
Runs `qwen3:8b` through a local Ollama server on the shared dev/test sets, scores it against the gold, and records a
hardware/latency check. **P5 only**: keep public figures, organizations, places, non-identifying dates; redact private persons,
contacts, IDs, street addresses, personal URLs, usernames, dates of birth, demographics.

Run **Sections 0-3 first** (data check + system check), then 4-10. Data (`data/processed/`) and results
(`llm_p5/results/`) are gitignored: never commit them.

## Section 0: Config

In [ ]:
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "llm_p5" else HERE          # repo root (works from llm_p5/ or the root)
LLM_DIR = ROOT / "llm_p5"
RESULTS_DIR, REFERENCE_DIR = LLM_DIR / "results", LLM_DIR / "reference"
CASES_PATH = LLM_DIR / "system_check_cases.jsonl"
DATA_DIR = ROOT / "data" / "processed"            # must contain dev.jsonl and test.jsonl (unzip the shared data here)

MODEL_TAG = "qwen3:8b"
OLLAMA_URL = "http://localhost:11434"
WINDOW, OVERLAP = 1200, 100          # characters; windows end at sentence boundaries
NUM_CTX, NUM_PREDICT = 4096, 900
DEADLINE = 120                       # seconds, wall clock per request
SEED = 42
MATCH_THRESHOLD = 1.0                # 1.0 = exact word-bounded string match (the only mode implemented)

RUN_SPLITS = ["dev"]                 # add "test" only after the prompt is frozen
TAB_SCOPE = "direct+quasi"           # "direct+quasi" | "direct": headline TAB gold scope (Sections 7, 8, 10)
TAB_ANNOTATOR = "first"              # "first" | "union"
SAMPLE_SIZES = {"crapii": {"dev": 150, "test": 150}, "tab": {"dev": 10, "test": 10}}
SAMPLE_IDS_FILE = None               # json list of doc_ids: overrides sampling (same docs as Presidio/GLiNER)
N_BOOT = 1000

> **Prompt freeze.** Leave `RUN_SPLITS = ["dev"]` while you iterate on the prompt. Add `"test"` only once the prompt is frozen:
> every result row records `prompt_hash`, and rows made with a different hash are never reused or mixed.

## Section 1: Helpers
Pure functions (cells tagged `helpers` are exec'd by `llm_p5/tests`). Nothing here talks to Ollama until called.

### 1a. Windows, string matching, merge

In [ ]:
import hashlib, http.client, json, os, platform, random, re, socket, subprocess, sys, threading, time, urllib.parse, urllib.request
import numpy as np
import pandas as pd

UNIFIED_LABELS = ["PERSON_PRIVATE", "PERSON_PUBLIC", "EMAIL", "PHONE", "ID_NUM", "STREET_ADDRESS", "DOB",
                  "DATE_OTHER", "LOCATION_OTHER", "URL_PERSONAL", "OTHER_PII", "DEMOGRAPHIC", "ORG"]
P5_KEEP = {"PERSON_PUBLIC", "ORG", "DATE_OTHER", "LOCATION_OTHER"}   # everything else is REDACT under P5_text

_SENT_END = re.compile(r"[.!?][\"')\]]*\s|\n")


def make_windows(text, size=1200, overlap=100):
    """Sentence-boundary windows [(start, end)] in document offsets; consecutive windows overlap by ~`overlap` chars."""
    n = len(text)
    if n == 0:
        return []
    out, start = [], 0
    while True:
        end = min(start + size, n)
        if end < n:
            lo = start + size // 2
            cut = None
            for m in _SENT_END.finditer(text, lo, end):
                cut = m.end()
            if cut is None:
                ws = max(text.rfind(c, lo, end) for c in (" ", "\n", "\t"))
                cut = ws + 1 if ws >= 0 else end
            end = cut
        out.append((start, end))
        if end >= n:
            break
        nxt = max(end - overlap, start + 1)
        while nxt < end and not text[nxt - 1].isspace():   # begin the next window at a word start
            nxt += 1
        start = nxt
    return out


def match_strings(wtext, items, offset=0):
    """Map returned strings to ALL word-bounded occurrences in the window, longest first, no overlaps.
    items: [{text,label,decision}]. Returns (preds in document offsets, n_unmatched_strings)."""
    best = {}
    for it in items:
        t = (it.get("text") or "").strip()
        if not t:
            continue
        cur = best.get(t)
        if cur is None or (it.get("decision") == "REDACT" and cur.get("decision") != "REDACT"):
            best[t] = {"label": it.get("label"), "decision": it.get("decision")}
    taken, preds, unmatched = [], [], 0
    for t in sorted(best, key=lambda x: (-len(x), x)):
        pat = re.compile(r"(?<!\w)" + re.escape(t) + r"(?!\w)")
        seen = False
        for m in pat.finditer(wtext):
            seen = True
            s, e = m.span()
            if any(s < e2 and s2 < e for s2, e2 in taken):
                continue
            taken.append((s, e))
            preds.append({"start": s + offset, "end": e + offset, "text": t, **best[t]})
        unmatched += not seen
    return sorted(preds, key=lambda p: p["start"]), unmatched


def merge_preds(preds):
    """Union across windows: identical (start, end) collapse, REDACT wins over KEEP."""
    d = {}
    for p in preds:
        k = (p["start"], p["end"])
        c = d.get(k)
        if c is None or (p["decision"] == "REDACT" and c["decision"] != "REDACT"):
            d[k] = p
    return sorted(d.values(), key=lambda p: (p["start"], p["end"]))


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

### 1b. Guardrailed Ollama client, hardware info

In [ ]:
REQUEST_TIMEOUT = 120


def _stream(body, url, timeout, t0, num_predict, out):
    """Stream one request into `out`; raises OSError subclasses on connection failures."""
    u = urllib.parse.urlsplit(url)
    conn = http.client.HTTPConnection(u.hostname, u.port, timeout=timeout)
    state = {"timed_out": False}
    sock = []  # http.client drops conn.sock for close-delimited responses, so keep our own reference

    def kill():  # watchdog: abort the connection at the deadline, even if the server sends nothing
        state["timed_out"] = True
        try:
            sock[0].shutdown(socket.SHUT_RDWR)
        except (OSError, IndexError):
            pass
        conn.close()

    parts, chunks = [], 0
    try:
        conn.connect()
        sock.append(conn.sock)
        timer = threading.Timer(max(timeout - (time.perf_counter() - t0), 0.01), kill)
        timer.daemon = True
        timer.start()
        try:
            conn.request("POST", u.path or "/", json.dumps(body).encode(), {"Content-Type": "application/json"})
            resp = conn.getresponse()
            if resp.status != 200:
                out["error"] = f"HTTP {resp.status}: {resp.reason}"
                return
            while True:
                line = resp.readline()
                if not line:
                    if not state["timed_out"] and out["done_reason"] is None:
                        out["error"] = "incomplete stream"
                    break
                obj = json.loads(line)
                if obj.get("error"):
                    out["error"] = f"server: {obj['error']}"
                    break
                parts.append((obj.get("message") or {}).get("content") or "")
                chunks += 1
                if obj.get("done"):
                    out["done_reason"] = obj.get("done_reason") or "stop"
                    out["eval_count"], out["eval_duration_ns"] = obj.get("eval_count"), obj.get("eval_duration")
                    out["load_duration_ns"] = obj.get("load_duration")
                    break
                if chunks > num_predict:  # a chunk is >= 1 token: the server ignored the cap
                    out["done_reason"] = "length"
                    break
        except (OSError, ValueError, http.client.HTTPException) as e:
            state["timed_out"] |= isinstance(e, TimeoutError)  # socket timeout racing the watchdog
            if not state["timed_out"]:
                out["error"] = f"{type(e).__name__}: {e}"
        finally:
            timer.cancel()
    finally:
        conn.close()
    out["raw"] = "".join(parts) or None
    if out["eval_count"] is None:
        out["eval_count"] = chunks
    if state["timed_out"]:
        out["error"] = "timeout"
    elif out["done_reason"] == "length":
        out["truncated"], out["error"] = True, "truncated"


def chat(model, messages, schema, url, num_ctx=4096, seed=42, timeout=REQUEST_TIMEOUT, num_predict=900):
    """POST to Ollama /api/chat (url = full endpoint). Streams internally, caps tokens, enforces a wall-clock deadline,
    think=false, temperature 0. No retry on timeout, truncation or bad JSON; one retry only on a refused connection.
    Returns {raw, parsed, valid_json, latency_s, eval_count, eval_duration_ns, load_duration_ns, tokens_per_s,
    error, done_reason, truncated}."""
    body = {
        "model": model, "messages": messages, "format": schema, "stream": True, "think": False,
        "keep_alive": "5m", "options": {"temperature": 0, "num_ctx": num_ctx, "seed": seed, "num_predict": num_predict},
    }
    out = {"raw": None, "parsed": None, "valid_json": False, "latency_s": None, "eval_count": None,
           "eval_duration_ns": None, "load_duration_ns": None, "tokens_per_s": None, "error": None,
           "done_reason": None, "truncated": False}
    t0 = time.perf_counter()
    for attempt in (0, 1):
        try:
            _stream(body, url, timeout, t0, num_predict, out)
            break
        except ConnectionRefusedError as e:
            out["error"] = f"{type(e).__name__}: {e}"
        except (OSError, http.client.HTTPException) as e:  # connect timeout, DNS, reset
            out["error"] = "timeout" if isinstance(e, TimeoutError) else f"{type(e).__name__}: {e}"
            break
    out["latency_s"] = time.perf_counter() - t0
    if out["eval_count"]:
        dur = (out["eval_duration_ns"] or 0) / 1e9 or out["latency_s"]
        out["tokens_per_s"] = out["eval_count"] / dur if dur else None
    if out["error"] is None and out["raw"] is not None:
        try:
            parsed = json.loads(out["raw"])
            if isinstance(parsed, dict):
                out["parsed"], out["valid_json"] = parsed, True
        except ValueError:
            pass
    return out


def make_chat_fn(model, base_url, num_ctx=4096, num_predict=900, deadline=120, seed=42):
    """Bind the settings; returns chat_fn(messages, schema) -> chat() result."""
    url = base_url.rstrip("/") + "/api/chat"
    return lambda messages, schema: chat(model, messages, schema, url, num_ctx=num_ctx, seed=seed,
                                         timeout=deadline, num_predict=num_predict)


def _http_json(url, data=None, timeout=10):
    req = urllib.request.Request(url, data=None if data is None else json.dumps(data).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())


def ollama_version(base_url):
    try:
        return _http_json(base_url.rstrip("/") + "/api/version").get("version")
    except Exception:
        return None


def model_info(tag, base_url):
    """{digest, size} for a pulled model tag (None values if the server is unreachable or the tag is missing)."""
    try:
        for m in _http_json(base_url.rstrip("/") + "/api/tags").get("models", []):
            if m.get("name") == tag or m.get("model") == tag or m.get("name") == tag + ":latest":
                return {"digest": m.get("digest"), "size": m.get("size")}
    except Exception:
        pass
    return {"digest": None, "size": None}


def gpu_share(tag, base_url):
    """size_vram / size of the loaded model from /api/ps (None if not loaded)."""
    try:
        for m in _http_json(base_url.rstrip("/") + "/api/ps").get("models", []):
            if m.get("name") == tag or m.get("model") == tag:
                return (m["size_vram"] / m["size"]) if m.get("size") else None
    except Exception:
        pass
    return None


def _sh(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=5).stdout.strip()
    except Exception:
        return ""


def hw_info():
    """OS, chip, RAM, Python. Deliberately no hostname, username or paths."""
    chip, ram = "", None
    if platform.system() == "Darwin":
        chip = _sh(["sysctl", "-n", "machdep.cpu.brand_string"])
        mem = _sh(["sysctl", "-n", "hw.memsize"])
        ram = round(int(mem) / 2**30) if mem.isdigit() else None
    elif platform.system() == "Linux":
        try:
            for line in open("/proc/cpuinfo"):
                if line.lower().startswith("model name"):
                    chip = line.split(":", 1)[1].strip()
                    break
            for line in open("/proc/meminfo"):
                if line.startswith("MemTotal"):
                    ram = round(int(line.split()[1]) / 2**20)
                    break
        except OSError:
            pass
    chip = chip or platform.processor() or platform.machine() or "unknown"
    return {"os": platform.platform(), "chip": chip, "ram_gb": ram, "python": platform.python_version()}


def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_") or "unknown"


def git_info():
    try:
        commit = _sh(["git", "rev-parse", "--short", "HEAD"]) or None
        dirty = bool(_sh(["git", "status", "--porcelain", "--untracked-files=no"]))
    except Exception:
        commit, dirty = None, None
    return {"git_commit": commit, "git_dirty": dirty}

### 1c. Prompts (P5 policy inlined) and prompt hash

In [ ]:
LABEL_DEFS = {
    "DOB": "a person's date of birth",
    "DATE_OTHER": "any other date",
    "PERSON_PUBLIC": "a well-known public or historical figure",
    "PERSON_PRIVATE": "any other person",
    "STREET_ADDRESS": "a home-level address",
    "LOCATION_OTHER": "city, region, venue or non-home place",
    "DEMOGRAPHIC": "age, gender or similar",
    "ORG": "an organization",
    "ID_NUM": "an identification number",
    "EMAIL": "an email address",
    "PHONE": "a phone number",
    "URL_PERSONAL": "a personal URL",
    "OTHER_PII": "any other personally identifying detail",
}
_DEFS = "\n".join(f"- {k}: {v}" for k, v in LABEL_DEFS.items())
_POLICY = """Policy (purpose P5, text that will be shared as ordinary text):
- KEEP: public or historical figures, organizations, cities, regions and venues, dates that do not identify a person.
- REDACT: private persons, email addresses, phone numbers, identification numbers, street or home addresses, personal URLs, usernames and handles, dates of birth, and demographic details (age, gender or similar) of an individual."""

SYSTEM_STANDALONE = f"""You are a privacy-redaction detector. Read the text and list every distinct person, place, organization, date, contact detail and identifier string in it, once each, exactly as written. For each string give one label and one decision, REDACT or KEEP, following the policy.

{_POLICY}

Labels:
{_DEFS}

Rules:
- Copy each string exactly as it appears in the text; do not paraphrase or add words.
- List each distinct string once, even if it appears several times.
- Do not give offsets or positions.
- If nothing qualifies, return an empty list.

Answer with JSON: {{"spans": [{{"text", "label", "decision"}}]}}."""

SYSTEM_REVIEW = f"""You are a privacy-redaction reviewer. You are given a text and a numbered list of candidate spans found by a detector. For each candidate give the decision REDACT or KEEP following the policy. You may also list personal-information strings in the text that are missing from the candidate list (copied exactly as written, with a label and a decision).

{_POLICY}

Labels:
{_DEFS}

Answer with JSON: {{"decisions": [{{"id", "decision"}}], "missed": [{{"text", "label", "decision"}}]}}."""

_DECISION = {"type": "string", "enum": ["REDACT", "KEEP"]}
_LABEL = {"type": "string", "enum": UNIFIED_LABELS}
SCHEMA_STANDALONE = {"type": "object", "required": ["spans"], "properties": {"spans": {"type": "array", "items": {
    "type": "object", "required": ["text", "label", "decision"],
    "properties": {"text": {"type": "string"}, "label": _LABEL, "decision": _DECISION}}}}}
SCHEMA_REVIEW = {"type": "object", "required": ["decisions", "missed"], "properties": {
    "decisions": {"type": "array", "items": {"type": "object", "required": ["id", "decision"],
                                             "properties": {"id": {"type": "integer"}, "decision": _DECISION}}},
    "missed": {"type": "array", "items": {"type": "object", "required": ["text", "label", "decision"],
                                          "properties": {"text": {"type": "string"}, "label": _LABEL,
                                                         "decision": _DECISION}}}}}


def prompt_hash(mode="standalone"):
    system, schema = (SYSTEM_REVIEW, SCHEMA_REVIEW) if mode == "review" else (SYSTEM_STANDALONE, SCHEMA_STANDALONE)
    return hashlib.sha1((system + json.dumps(schema, sort_keys=True)).encode()).hexdigest()[:12]


def build_messages(window_text):
    return [{"role": "system", "content": SYSTEM_STANDALONE}, {"role": "user", "content": f"TEXT:\n{window_text}"}]


def build_review_messages(window_text, cands):
    """cands: [{text,label,score}] in display order (ids are 1..k). Only text/label/score are shown: no gold info."""
    rows = ["id | text | detector_label | score"]
    for i, c in enumerate(cands, 1):
        score = c.get("score")
        rows.append(f"{i} | {c.get('text')} | {c.get('label')} | {'' if score is None else round(float(score), 3)}")
    user = f"TEXT:\n{window_text}\n\nCANDIDATES:\n" + "\n".join(rows)
    return [{"role": "system", "content": SYSTEM_REVIEW}, {"role": "user", "content": user}]

### 1d. Detection (resume-safe)

In [ ]:
def _items(parsed, key="spans"):
    """Valid span items from a parsed reply, or None when the reply has the wrong shape."""
    lst = parsed.get(key) if isinstance(parsed, dict) else None
    if not isinstance(lst, list):
        return None
    return [x for x in lst if isinstance(x, dict) and isinstance(x.get("text"), str)
            and x.get("decision") in ("REDACT", "KEEP")]


def detect_doc(doc, chat_fn, window=1200, overlap=100, mode="standalone", candidates=None):
    """Run the LLM over every window of one doc; returns a result row (no metadata)."""
    text = doc["text"]
    wins = make_windows(text, window, overlap)
    cands = sorted(candidates or [], key=lambda c: (c["start"], c["end"]))
    reviewed = set()
    preds, wstats, errors = [], [], {}
    n_trunc = n_inv = n_unm = 0
    t0 = time.perf_counter()
    for s, e in wins:
        wt = text[s:e]
        if mode == "review":
            idx = [i for i, c in enumerate(cands) if c["start"] >= s and c["end"] <= e]
            msgs, schema = build_review_messages(wt, [cands[i] for i in idx]), SCHEMA_REVIEW
        else:
            msgs, schema = build_messages(wt), SCHEMA_STANDALONE
        r = chat_fn(msgs, schema)
        ok = bool(r.get("valid_json")) and not r.get("truncated")
        items = _items(r.get("parsed"), "missed" if mode == "review" else "spans") if ok else None
        if r.get("truncated"):
            n_trunc += 1
        elif items is None:
            n_inv += 1
        if r.get("error"):
            errors[r["error"]] = errors.get(r["error"], 0) + 1
        wstats.append({"latency_s": r.get("latency_s"), "tokens_per_s": r.get("tokens_per_s"),
                       "eval_count": r.get("eval_count"), "valid": items is not None,
                       "truncated": bool(r.get("truncated")),
                       "load_s": (r.get("load_duration_ns") or 0) / 1e9})
        if items is None:
            continue
        if mode == "review":
            dec = {d.get("id"): d.get("decision") for d in _items_dec(r["parsed"])}
            for k, i in enumerate(idx, 1):
                c = cands[i]
                reviewed.add(i)
                preds.append({"start": c["start"], "end": c["end"], "text": c.get("text", text[c["start"]:c["end"]]),
                              "label": c.get("label"), "decision": dec.get(k, "REDACT")})
        p, u = match_strings(wt, items, s)
        preds += p
        n_unm += u
    if mode == "review":   # candidates no window fully contained: keep the detector's call (REDACT)
        for i, c in enumerate(cands):
            if i not in reviewed:
                preds.append({"start": c["start"], "end": c["end"], "text": c.get("text", ""),
                              "label": c.get("label"), "decision": "REDACT"})
    return {"doc_id": doc["doc_id"], "source": doc.get("source"), "split": doc.get("split"), "mode": mode,
            "n_windows": len(wins), "n_truncated": n_trunc, "n_invalid": n_inv, "n_unmatched": n_unm,
            "preds": merge_preds(preds), "latency_s": time.perf_counter() - t0, "windows": wstats, "errors": errors}


def _items_dec(parsed):
    lst = parsed.get("decisions") if isinstance(parsed, dict) else None
    return [x for x in (lst or []) if isinstance(x, dict) and x.get("decision") in ("REDACT", "KEEP")]


def _row_key(r):
    return (r.get("doc_id"), r.get("mode"), r.get("prompt_hash"), r.get("model_digest"), r.get("window"), r.get("overlap"))


def load_rows(path, meta, mode, window, overlap):
    """Rows already on disk that were produced with the same prompt hash, model digest and window settings."""
    done = {}
    if os.path.exists(path):
        for r in load_jsonl(path):
            if _row_key(r) == (r.get("doc_id"), mode, meta["prompt_hash"], meta.get("model_digest"), window, overlap):
                done[r["doc_id"]] = r
    return done


def run_detection(docs, out_path, chat_fn, meta, mode="standalone", candidates_by_doc=None,
                  window=1200, overlap=100, log=print):
    """Resume-safe: one JSONL row per doc appended to out_path. meta: prompt_hash, model_digest, git_*, model_tag."""
    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
    done = load_rows(out_path, meta, mode, window, overlap)
    todo = [d for d in docs if d["doc_id"] not in done]
    log(f"{len(done)} docs already done, {len(todo)} to run -> {out_path}")
    t0 = time.perf_counter()
    for i, d in enumerate(todo, 1):
        row = detect_doc(d, chat_fn, window, overlap, mode, (candidates_by_doc or {}).get(d["doc_id"]))
        if row["n_invalid"] + row["n_truncated"] == row["n_windows"] and any("ConnectionRefused" in k for k in row["errors"]):
            raise RuntimeError("Ollama refused the connection: is `ollama serve` running? (no row written)")
        row.update({k: meta.get(k) for k in ("prompt_hash", "model_digest", "git_commit", "git_dirty")})
        row.update({"model_tag": meta.get("model_tag"), "window": window, "overlap": overlap})
        with open(out_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row) + "\n")
        done[d["doc_id"]] = row
        log(f"[{i}/{len(todo)}] {d['doc_id']} {d.get('source')} win={row['n_windows']} trunc={row['n_truncated']} "
            f"inv={row['n_invalid']} unm={row['n_unmatched']} preds={len(row['preds'])} {row['latency_s']:.1f}s")
        if i % 10 == 0:
            el = time.perf_counter() - t0
            log(f"   ETA ~{el / i * (len(todo) - i) / 60:.1f} min for the remaining {len(todo) - i} docs")
    return [done[d["doc_id"]] for d in docs]

### 1e. Gold adapters

In [ ]:
def _span(s, decision, label=None):
    return {"start": s["start"], "end": s["end"], "label": label or s.get("label"), "decision": decision}


def gold_crapii(doc):
    """CRAPII: every gold span is redact."""
    return [_span(s, "redact") for s in doc["spans"]]


def gold_tab(doc, scope="direct+quasi", annotator="first"):
    """TAB. scope 'direct+quasi': DIRECT, QUASI = redact; NO_MASK = keep.
    scope 'direct': DIRECT = redact, QUASI = ignore, NO_MASK = keep. MISC/QUANTITY (ignored_spans) = ignore.
    annotator 'first' = primary spans only; 'union' = primary plus every other annotator."""
    spans = list(doc["spans"])
    if annotator == "union":
        for lst in (doc.get("other_annotations") or {}).values():
            spans += lst
    out = []
    for s in spans:
        it = (s.get("attrs") or {}).get("identifier_type")
        if s.get("label") not in UNIFIED_LABELS:
            out.append(_span(s, "ignore", s.get("label")))
        elif it == "NO_MASK":
            out.append(_span(s, "keep"))
        elif it == "QUASI" and scope == "direct":
            out.append(_span(s, "ignore"))
        else:
            out.append(_span(s, "redact"))
    out += [_span(s, "ignore", s.get("label")) for s in doc.get("ignored_spans") or []]
    return out


def expected_decision(label):
    return "KEEP" if label in P5_KEEP else "REDACT"


def gold_synthetic(case):
    """Synthetic system-check cases: spans carry expected_decision."""
    return [_span(s, "redact" if s["expected_decision"] == "REDACT" else "keep", s.get("gold_label"))
            for s in case["spans"]]


def gold_for(doc, scope="direct+quasi", annotator="first"):
    src = doc.get("source")
    if src == "tab":
        return gold_tab(doc, scope, annotator)
    if src == "synthetic":
        return gold_synthetic(doc)
    return gold_crapii(doc)

### 1f. Metrics, bootstrap CIs, error examples

In [ ]:
def _mask(n, spans, nz):
    m = np.zeros(n, bool)
    for s, e in spans:
        m[max(s, 0):min(e, n)] = True
    return m & nz


def _dedupe(gold):
    seen, out = set(), []
    for g in gold:
        k = (g["start"], g["end"], g["label"], g["decision"])
        if k not in seen:
            seen.add(k)
            out.append(g)
    return out


def doc_counts(text, gold, preds):
    """Raw counts for one doc (whitespace excluded everywhere). Priority of gold regions: redact > keep > ignore.
    Only REDACT predictions count (missing decision = REDACT); predictions fully inside ignore regions are dropped."""
    n = len(text)
    nz = np.fromiter((not c.isspace() for c in text), bool, n)
    gold = _dedupe(gold)
    red = [g for g in gold if g["decision"] == "redact"]
    keep = [g for g in gold if g["decision"] == "keep"]
    R = _mask(n, [(g["start"], g["end"]) for g in red], nz)
    K = _mask(n, [(g["start"], g["end"]) for g in keep], nz) & ~R
    I = _mask(n, [(g["start"], g["end"]) for g in gold if g["decision"] == "ignore"], nz) & ~R & ~K
    redp = [p for p in preds if p.get("decision", "REDACT") == "REDACT" and p["end"] > p["start"]]
    C = _mask(n, [(p["start"], p["end"]) for p in redp], nz)          # all REDACT chars: used for coverage
    kept = []
    for p in redp:
        s, e = max(p["start"], 0), min(p["end"], n)
        seg = nz[s:e] & ~I[s:e]
        if seg.any():                                                   # not whitespace-only, not fully ignored
            kept.append((s, e, not (seg & ~R[s:e]).any()))
    F = _mask(n, [(s, e) for s, e, _ in kept], nz) & ~I
    c = {"n_docs": 1, "pred_n": len(kept), "pred_tp": sum(t for _, _, t in kept),
         "flag_chars": int(F.sum()), "flag_nonpii": int((F & ~R).sum()),
         "nonpii_chars": int((nz & ~R & ~I).sum()), "gold_chars": int(R.sum()), "gold_uncov": int((R & ~C).sum())}
    rec_n = rec_c = 0
    for g in red:
        gm = _mask(n, [(g["start"], g["end"])], nz)
        if not gm.any():
            continue
        cov = not (gm & ~C).any()
        rec_n += 1
        rec_c += cov
        c[f"rec_n|{g['label']}"] = c.get(f"rec_n|{g['label']}", 0) + 1
        c[f"rec_c|{g['label']}"] = c.get(f"rec_c|{g['label']}", 0) + cov
    keep_n = keep_hit = 0
    for g in keep:
        gm = _mask(n, [(g["start"], g["end"])], nz) & ~R
        if not gm.any():
            continue
        hit = bool((gm & C).any())
        keep_n += 1
        keep_hit += hit
        c[f"keep_n|{g['label']}"] = c.get(f"keep_n|{g['label']}", 0) + 1
        c[f"keep_hit|{g['label']}"] = c.get(f"keep_hit|{g['label']}", 0) + hit
    has_gold = rec_n > 0
    c.update({"rec_n": rec_n, "rec_c": rec_c, "keep_n": keep_n, "keep_hit": keep_hit,
              "docs_gold": int(has_gold), "docs_leaked": int(has_gold and rec_c < rec_n),
              "docs_nogold": int(not has_gold), "docs_false_alarm": int((not has_gold) and len(kept) > 0)})
    return c


def _div(a, b):
    return float(a) / b if b else float("nan")


def metrics_from_sums(s):
    """Aggregate metrics from summed doc_counts (a dict; absent keys = 0)."""
    g = lambda k: float(s.get(k, 0))
    P, R = _div(g("pred_tp"), g("pred_n")), _div(g("rec_c"), g("rec_n"))
    p, r = (0.0 if P != P else P), (0.0 if R != R else R)
    m = {"span_recall": R, "span_precision": P,
         "f2": _div(5 * p * r, 4 * p + r),
         "char_over_redaction_pct": 100 * _div(g("flag_nonpii"), g("flag_chars")),
         "non_pii_flag_rate": _div(g("flag_nonpii"), g("nonpii_chars")),
         "char_leakage_pct": 100 * _div(g("gold_uncov"), g("gold_chars")),
         "doc_leakage": _div(g("docs_leaked"), g("docs_gold")),
         "doc_false_alarm": _div(g("docs_false_alarm"), g("docs_nogold")),
         "keep_false_redaction_rate": _div(g("keep_hit"), g("keep_n"))}
    for k in s:
        if k.startswith("rec_n|"):
            lab = k.split("|", 1)[1]
            m[f"recall|{lab}"] = _div(g(f"rec_c|{lab}"), g(k))
        elif k.startswith("keep_n|"):
            lab = k.split("|", 1)[1]
            m[f"keep_false|{lab}"] = _div(g(f"keep_hit|{lab}"), g(k))
    return m


def bootstrap_ci(counts, n_boot=1000, seed=42):
    """95% percentile CIs over documents. counts: DataFrame, one row per doc."""
    cols, M = list(counts.columns), counts.to_numpy(float)
    rng = np.random.default_rng(seed)
    N = len(M)
    reps = [metrics_from_sums(dict(zip(cols, M[rng.integers(0, N, N)].sum(0)))) for _ in range(n_boot)]
    ci = {}
    for k in reps[0]:
        v = np.array([r[k] for r in reps])
        v = v[~np.isnan(v)]
        ci[k] = tuple(np.percentile(v, [2.5, 97.5])) if len(v) else (float("nan"), float("nan"))
    return ci


HEADLINE = ["span_recall", "span_precision", "f2", "char_over_redaction_pct", "non_pii_flag_rate",
            "char_leakage_pct", "doc_leakage", "doc_false_alarm", "keep_false_redaction_rate"]


def score_table(rows, docs_by_id, scope="direct+quasi", annotator="first", sources=None, n_boot=1000, seed=42):
    """Per (source, split) metrics with 95% CIs. Returns (summary_df, label_df). Sources are never pooled."""
    groups = {}
    for r in rows:
        d = docs_by_id.get(r["doc_id"])
        if d is None or (sources and d["source"] not in sources):
            continue
        groups.setdefault((d["source"], d.get("split")), []).append(
            doc_counts(d["text"], gold_for(d, scope, annotator), r["preds"]))
    summ, lab = [], []
    for (src, sp), cl in sorted(groups.items(), key=lambda kv: (str(kv[0][0]), str(kv[0][1]))):
        df = pd.DataFrame(cl).fillna(0)
        m, ci = metrics_from_sums(df.sum().to_dict()), bootstrap_ci(df, n_boot, seed)
        base = {"source": src, "split": sp, "scope": scope if src == "tab" else "n/a",
                "annotator": annotator if src == "tab" else "n/a"}
        row = {**base, "n_docs": len(df), "n_gold_redact": int(df.get("rec_n", pd.Series([0])).sum()),
               "n_gold_keep": int(df.get("keep_n", pd.Series([0])).sum())}
        for k in HEADLINE:
            row[k], row[k + "_lo"], row[k + "_hi"] = m[k], ci[k][0], ci[k][1]
        summ.append(row)
        for k, v in m.items():
            if "|" in k:
                kind, label = k.split("|", 1)
                nk = ("rec_n|" if kind == "recall" else "keep_n|") + label
                lab.append({**base, "kind": kind, "label": label, "n": int(df[nk].sum()), "value": v,
                            "lo": ci[k][0], "hi": ci[k][1], "n_docs": len(df)})
    return pd.DataFrame(summ), pd.DataFrame(lab)


def fmt_ci(df, cols=HEADLINE):
    """Display copy: 'value [lo, hi]' strings."""
    out = df[["source", "split", "scope", "n_docs", "n_gold_redact", "n_gold_keep"]].copy()
    for k in cols:
        out[k] = [f"{v:.3f} [{lo:.3f}, {hi:.3f}]" for v, lo, hi in zip(df[k], df[k + "_lo"], df[k + "_hi"])]
    return out


def context(text, s, e, width=120):
    if e - s >= width:
        return text[s:s + width]
    pad = (width - (e - s)) // 2
    return text[max(0, s - pad):e + pad][:width]


def worst_errors(rows, docs_by_id, scope="direct+quasi", annotator="first", k=10):
    """Worst-k per source: gold redact spans missed (most uncovered chars) and keep spans falsely redacted."""
    miss, fals = [], []
    for r in rows:
        d = docs_by_id.get(r["doc_id"])
        if d is None:
            continue
        text, n = d["text"], len(d["text"])
        nz = np.fromiter((not c.isspace() for c in text), bool, n)
        gold = _dedupe(gold_for(d, scope, annotator))
        redp = [(p["start"], p["end"]) for p in r["preds"] if p.get("decision", "REDACT") == "REDACT"]
        C = _mask(n, redp, nz)
        R = _mask(n, [(g["start"], g["end"]) for g in gold if g["decision"] == "redact"], nz)
        for g in gold:
            gm = _mask(n, [(g["start"], g["end"])], nz)
            base = {"source": d["source"], "split": d.get("split"), "doc_id": d["doc_id"], "label": g["label"],
                    "text": text[g["start"]:g["end"]][:60], "context": context(text, g["start"], g["end"])}
            if g["decision"] == "redact" and gm.any() and (gm & ~C).any():
                miss.append({**base, "uncovered_chars": int((gm & ~C).sum())})
            elif g["decision"] == "keep" and (gm & ~R).any() and (gm & ~R & C).any():
                fals.append({**base, "overlap_chars": int((gm & ~R & C).sum())})
    def top(lst, key):
        df = pd.DataFrame(lst)
        if df.empty:
            return df
        return df.sort_values(key, ascending=False).groupby("source", sort=False).head(k).reset_index(drop=True)
    return top(miss, "uncovered_chars"), top(fals, "overlap_chars")


def load_baseline(path):
    """Baseline predictions JSONL {doc_id, spans:[{start,end,text,label,decision?}]} -> rows (missing decision = REDACT)."""
    rows = []
    for r in load_jsonl(path):
        spans = [{**s, "decision": s.get("decision", "REDACT")} for s in r["spans"]]
        rows.append({"doc_id": r["doc_id"], "preds": spans})
    return rows

### 1g. System-check, sampling, fingerprint helpers

In [ ]:
def decision_agreement(gold_spans, preds):
    """Decision-level agreement with expected_decision. gold_spans: [{start,end,expected_decision}].
    A gold span is predicted REDACT if any REDACT prediction overlaps it, otherwise KEEP."""
    red = [(p["start"], p["end"]) for p in preds if p.get("decision", "REDACT") == "REDACT"]
    agree, bad = 0, []
    for g in gold_spans:
        got = "REDACT" if any(s < g["end"] and g["start"] < e for s, e in red) else "KEEP"
        if got == g["expected_decision"]:
            agree += 1
        else:
            bad.append({"text": g.get("text"), "expected": g["expected_decision"], "got": got})
    return agree, len(gold_spans), bad


def pred_set(preds):
    return sorted({(p["text"], p["decision"]) for p in preds})


def set_agreement(a, b):
    """Jaccard agreement (%) of two (text, decision) sets; both empty = 100."""
    a, b = {tuple(x) for x in a}, {tuple(x) for x in b}
    return 100.0 * len(a & b) / len(a | b) if (a | b) else 100.0


def latency_stats(rows):
    """Stats over all windows of rows in run order, excluding the very first call (it includes model load)."""
    flat = [w for r in rows for w in r["windows"]]
    if not flat:
        return {}
    first, rest = flat[0], flat[1:]
    med = lambda xs: float(np.median(xs)) if xs else float("nan")
    return {"n_windows": len(flat), "first_call_latency_s": first["latency_s"], "model_load_s": first.get("load_s"),
            "median_latency_s": med([w["latency_s"] for w in rest if w["latency_s"] is not None]),
            "median_tokens_per_s": med([w["tokens_per_s"] for w in rest if w["tokens_per_s"]]),
            "valid_json_rate": (sum(w["valid"] for w in rest) / len(rest)) if rest else float("nan"),
            "n_truncated": sum(w["truncated"] for w in rest)}


def project_runtime(docs, median_s, window=1200, overlap=100):
    n = sum(len(make_windows(d["text"], window, overlap)) for d in docs)
    return {"n_docs": len(docs), "n_windows": n, "est_seconds": n * median_s}


def sample_docs(docs, n, stratified=False, seed=42):
    """Seeded doc-level sample. stratified: half the docs have >= 1 gold span (fills from the other group if short)."""
    rng = random.Random(seed)
    pool = sorted(docs, key=lambda d: d["doc_id"])
    if len(pool) <= n:
        return pool
    if not stratified:
        return sorted(rng.sample(pool, n), key=lambda d: d["doc_id"])
    pos, neg = [d for d in pool if d["spans"]], [d for d in pool if not d["spans"]]
    k_pos = min(n // 2, len(pos))
    k_neg = min(n - k_pos, len(neg))
    k_pos = min(n - k_neg, len(pos))
    return sorted(rng.sample(pos, k_pos) + rng.sample(neg, k_neg), key=lambda d: d["doc_id"])


def data_fingerprint(data_dir, splits=("dev", "test")):
    fp = {}
    for sp in splits:
        path = os.path.join(data_dir, f"{sp}.jsonl")
        h, rows, per = hashlib.sha256(), 0, {}
        with open(path, "rb") as f:
            for line in f:
                h.update(line)
                if line.strip():
                    rows += 1
                    src = json.loads(line).get("source")
                    per[src] = per.get(src, 0) + 1
        fp[f"{sp}.jsonl"] = {"sha256": h.hexdigest(), "rows": rows, "docs_per_source": per}
    return fp

## Section 2: Data check

In [ ]:
need = [DATA_DIR / "dev.jsonl", DATA_DIR / "test.jsonl"]
missing = [str(p.relative_to(ROOT)) for p in need if not p.exists()]
assert not missing, (f"Missing {missing}. Unzip the shared data folder so that dev.jsonl and test.jsonl exist in "
                     f"{DATA_DIR.relative_to(ROOT)}/ (the data is gitignored and comes from the team zip).")
FINGERPRINT = data_fingerprint(DATA_DIR)
for name, fp in FINGERPRINT.items():
    print(f"{name}: rows={fp['rows']} docs_per_source={fp['docs_per_source']}\n   sha256={fp['sha256']}")
ref_fp = REFERENCE_DIR / "data_fingerprint.json"
if ref_fp.exists():
    ref = json.loads(ref_fp.read_text())
    for name, fp in FINGERPRINT.items():
        if ref.get(name, {}).get("sha256") != fp["sha256"]:
            print(f"WARNING: {name} differs from llm_p5/reference/data_fingerprint.json: you may not have the shared data version.")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data_fingerprint.json").write_text(json.dumps(FINGERPRINT, indent=1))

## Section 3: System check
30 fictional cases (committed, safe to share) through the same detector. First call excluded from stats (it includes model load).
Needs Ollama running and the model pulled. The runtime projection for your sample sizes is printed in Section 4.

In [ ]:
HW = hw_info()
INFO = model_info(MODEL_TAG, OLLAMA_URL)
OLLAMA_VER = ollama_version(OLLAMA_URL)
assert INFO["digest"], f"{MODEL_TAG} not found: start Ollama and run `ollama pull {MODEL_TAG}`."
PHASH = prompt_hash("standalone")
META = {"prompt_hash": PHASH, "model_digest": INFO["digest"], "model_tag": MODEL_TAG, **git_info()}
chat_fn = make_chat_fn(MODEL_TAG, OLLAMA_URL, NUM_CTX, NUM_PREDICT, DEADLINE, SEED)

cases = load_jsonl(CASES_PATH)
chk_docs = [{"doc_id": c["case_id"], "source": "synthetic", "split": "check", "text": c["original"], "spans": c["spans"]}
            for c in cases]
chk_path = RESULTS_DIR / "system_check_rows.jsonl"
chk_path.unlink(missing_ok=True)                       # always a fresh timing run
chk_rows = run_detection(chk_docs, chk_path, chat_fn, META, "standalone", None, WINDOW, OVERLAP)
GPU = gpu_share(MODEL_TAG, OLLAMA_URL)

STATS = latency_stats(chk_rows)
per_case, ag, tot = {}, 0, 0
for d, r in zip(chk_docs, chk_rows):
    a, t, bad = decision_agreement(d["spans"], r["preds"])
    ag, tot = ag + a, tot + t
    per_case[d["doc_id"]] = {"agreement": a / t if t else None, "mismatches": bad, "pred_set": pred_set(r["preds"])}
STATS["decision_agreement"] = ag / tot if tot else float("nan")
print(f"model load (first call): {STATS['model_load_s']:.1f}s, first call total {STATS['first_call_latency_s']:.1f}s (excluded from stats)")
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in STATS.items()})

chip, ram = slug(HW["chip"]), HW["ram_gb"]
CHECK_PATH = RESULTS_DIR / f"system_check_{chip}_{ram}gb.json"
REPORT = {"hardware": HW, "ollama_version": OLLAMA_VER, "model": {"tag": MODEL_TAG, **INFO}, "gpu_share": GPU,
          "prompt_hash": PHASH, "config": {"window": WINDOW, "overlap": OVERLAP, "num_ctx": NUM_CTX,
          "num_predict": NUM_PREDICT, "deadline_s": DEADLINE, "seed": SEED},
          "stats": STATS, "cases": per_case, "data_fingerprint": FINGERPRINT}
CHECK_PATH.write_text(json.dumps(REPORT, indent=1))
print("wrote", CHECK_PATH.relative_to(ROOT))

In [ ]:
import pandas as pd
ref_path = REFERENCE_DIR / "system_check_reference.json"
if ref_path.exists():
    REF = json.loads(ref_path.read_text())
    rs = REF["stats"]
    tbl = pd.DataFrame({
        "this machine": [STATS["median_latency_s"], STATS["median_tokens_per_s"], GPU, STATS["valid_json_rate"]],
        "reference": [rs.get("median_latency_s"), rs.get("median_tokens_per_s"), REF.get("gpu_share"), rs.get("valid_json_rate")],
    }, index=["median latency / window (s)", "tokens / s", "GPU share", "valid JSON rate"])
    print(tbl.round(3).to_string())
    print("digest match:", INFO["digest"] == REF["model"]["digest"])
    if INFO["digest"] != REF["model"]["digest"]:
        print("WARNING: model digest differs from the reference: results are not comparable. Re-pull the exact model.")
    if REF.get("prompt_hash") != PHASH:
        print("WARNING: prompt hash differs from the reference run.")
    agr = {cid: set_agreement(per_case[cid]["pred_set"], REF["cases"][cid]["pred_set"])
           for cid in per_case if cid in REF.get("cases", {})}
    if agr:
        print(f"agreement of predicted (text, decision) sets with the reference: mean {np.mean(list(agr.values())):.1f}% "
              f"over {len(agr)} cases; lowest:", sorted(agr.items(), key=lambda kv: kv[1])[:5])
else:
    print("No llm_p5/reference/system_check_reference.json: nothing to compare against.")

## Section 4: Load and sample the splits

In [ ]:
DOCS = {sp: load_jsonl(DATA_DIR / f"{sp}.jsonl") for sp in RUN_SPLITS}
if SAMPLE_IDS_FILE:
    keep_ids = set(json.loads(Path(SAMPLE_IDS_FILE).read_text()))
    SELECTED = {sp: [d for d in ds if d["doc_id"] in keep_ids] for sp, ds in DOCS.items()}
else:
    SELECTED = {sp: [d for src in SAMPLE_SIZES
                     for d in sample_docs([x for x in ds if x["source"] == src], SAMPLE_SIZES[src][sp],
                                          stratified=(src == "crapii"), seed=SEED)]
                for sp, ds in DOCS.items()}
DOCS_BY_ID = {d["doc_id"]: d for ds in SELECTED.values() for d in ds}
med = STATS["median_latency_s"]
for sp, ds in SELECTED.items():
    for src in sorted({d["source"] for d in ds}):
        sub = [d for d in ds if d["source"] == src]
        pr = project_runtime(sub, med, WINDOW, OVERLAP)
        print(f"{sp}/{src}: {pr['n_docs']} docs ({sum(1 for d in sub if d['spans'])} with gold), {pr['n_windows']} windows, "
              f"estimated ~{pr['est_seconds'] / 60:.0f} min (estimate)")
tot_est = sum(project_runtime(ds, med, WINDOW, OVERLAP)["est_seconds"] for ds in SELECTED.values())
print(f"total estimated runtime ~{tot_est / 60:.0f} min (estimate, not a promise)")

## Section 5: Run Qwen standalone detection (resumable)
One JSONL row per doc in `llm_p5/results/`. Interrupt and re-run any time: finished docs are skipped.

In [ ]:
ROWS = {}
for sp in RUN_SPLITS:
    ROWS[sp] = run_detection(SELECTED[sp], RESULTS_DIR / f"qwen_standalone_{sp}.jsonl", chat_fn, META,
                             "standalone", None, WINDOW, OVERLAP)
ALL_ROWS = [r for rs in ROWS.values() for r in rs]
tot = lambda k: sum(r[k] for r in ALL_ROWS)
print(f"docs={len(ALL_ROWS)} windows={tot('n_windows')} truncated={tot('n_truncated')} invalid={tot('n_invalid')} "
      f"unmatched strings={tot('n_unmatched')}")

## Section 6: Metrics
Whitespace excluded. *Span recall* = gold redact spans fully covered by REDACT predictions; *span precision* = REDACT predictions fully
inside gold redact spans; *F2* = 5PR/(4P+R); *char over-redaction %* = non-PII chars flagged / all chars flagged;
*non_pii_flag_rate* = non-PII chars flagged / all non-PII chars; *char leakage %* = uncovered gold chars / all gold chars;
*doc leakage* = docs with an incompletely covered gold span / docs with gold PII; *doc false alarm* = docs without gold PII
that got a flag / docs without gold PII; *keep_false_redaction_rate* = gold keep spans overlapped by a REDACT prediction / gold keep spans.
Sources are never pooled; 95% bootstrap CIs over documents (1000, seed 42).

In [ ]:
TABLES, LABEL_TABLES = {}, {}
for scope in ["direct+quasi", "direct"]:
    s, l = score_table(ALL_ROWS, DOCS_BY_ID, scope, TAB_ANNOTATOR, sources=["tab"] if TABLES else None, n_boot=N_BOOT, seed=SEED)
    TABLES[scope], LABEL_TABLES[scope] = s, l
    print(f"=== scope={scope} (applies to TAB; CRAPII is scope-free), annotator={TAB_ANNOTATOR} ===")
    print(fmt_ci(s).to_string(index=False))

In [ ]:
for scope, l in LABEL_TABLES.items():
    print(f"=== recall by gold label / keep_false_redaction_rate by gold label, scope={scope} ===")
    out = l.assign(value=lambda x: x.value.round(3), lo=lambda x: x.lo.round(3), hi=lambda x: x.hi.round(3))
    print(out[["source", "split", "kind", "label", "n", "value", "lo", "hi"]].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, scope in zip(axes, TABLES):
    t = TABLES[scope]
    names = [f"{a}/{b}" for a, b in zip(t.source, t.split)]
    for off, k in zip((-0.2, 0.2), ("span_recall", "span_precision")):
        ax.bar([i + off for i in range(len(t))], t[k], 0.4, label=k,
               yerr=[t[k] - t[k + "_lo"], t[k + "_hi"] - t[k]], capsize=3)
    ax.set_xticks(range(len(t)), names)
    ax.set_title(f"TAB scope: {scope}")
axes[0].legend(); axes[0].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

## Section 7: Error examples
Worst-10 per source (context truncated to 120 chars). Misses = gold redact spans not covered; false redactions = gold keep spans hit by a REDACT prediction.

In [ ]:
MISSES, FALSE_RED = worst_errors(ALL_ROWS, DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR, k=10)
pd.set_option("display.max_colwidth", 120)
print("--- worst misses ---");           print(MISSES.drop(columns=["split"], errors="ignore").to_string(index=False))
print("--- worst false redactions of keep spans ---"); print(FALSE_RED.drop(columns=["split"], errors="ignore").to_string(index=False))

## Section 8: Optional: score a baseline predictions file
JSONL, one doc per line: `{doc_id, spans:[{start,end,text,label,decision?}]}` (e.g. GLiNER output; missing `decision` = REDACT).
Scored on the same docs as the Qwen run with the same metrics.

In [ ]:
BASELINE_PATH, BASELINE_NAME = None, "baseline"      # e.g. "results/preds/gliner_dev.jsonl"
SIDE = None
if BASELINE_PATH:
    base_rows = [r for r in load_baseline(BASELINE_PATH) if r["doc_id"] in DOCS_BY_ID]
    have = {r["doc_id"] for r in base_rows}
    print(f"baseline covers {len(have)}/{len(DOCS_BY_ID)} Qwen-run docs; scoring only the shared docs")
    shared = [r for r in ALL_ROWS if r["doc_id"] in have]
    q, _ = score_table(shared, DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR, n_boot=N_BOOT, seed=SEED)
    b, _ = score_table(base_rows, DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR, n_boot=N_BOOT, seed=SEED)
    SIDE = pd.concat([fmt_ci(q).assign(system=MODEL_TAG), fmt_ci(b).assign(system=BASELINE_NAME)]).sort_values(["source", "split", "system"])
    print(SIDE.to_string(index=False))
else:
    print("BASELINE_PATH not set: skipped.")

## Section 9: Optional review mode (disabled by default; architecture undecided)
A candidates JSONL `{doc_id, spans:[{start,end,text,label,score}]}` is shown per window as numbered candidates; the LLM returns
REDACT/KEEP per id and may list missed strings. No gold information is in the prompt. Rows carry `mode: "review"`.

In [ ]:
REVIEW_ENABLED, CANDIDATES_PATH = False, None
if REVIEW_ENABLED and CANDIDATES_PATH:
    cand = {r["doc_id"]: r["spans"] for r in load_jsonl(CANDIDATES_PATH)}
    rmeta = {**META, "prompt_hash": prompt_hash("review")}
    REVIEW_ROWS = []
    for sp in RUN_SPLITS:
        REVIEW_ROWS += run_detection(SELECTED[sp], RESULTS_DIR / f"qwen_review_{sp}.jsonl", chat_fn, rmeta,
                                     "review", cand, WINDOW, OVERLAP)
    s, _ = score_table(REVIEW_ROWS, DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR, n_boot=N_BOOT, seed=SEED)
    print(fmt_ci(s).to_string(index=False))
else:
    print("Review mode disabled.")

## Section 10: Export

In [ ]:
tag = f"{slug(HW['chip'])}_{HW['ram_gb']}gb"
frames = [TABLES["direct+quasi"].assign(scope_key="direct+quasi")]
frames.append(TABLES["direct"][TABLES["direct"].source == "tab"].assign(scope_key="direct"))
CSV = pd.concat(frames).assign(model=MODEL_TAG, prompt_hash=PHASH, annotator=TAB_ANNOTATOR)
csv_path = RESULTS_DIR / f"summary_{tag}.csv"
CSV.to_csv(csv_path, index=False)

lines = [f"# P5 Qwen summary ({tag})", "",
         f"- model `{MODEL_TAG}` digest `{INFO['digest']}`, prompt_hash `{PHASH}`, Ollama {OLLAMA_VER}",
         f"- hardware: {HW['chip']}, {HW['ram_gb']} GB RAM, {HW['os']}, Python {HW['python']}",
         f"- splits run: {RUN_SPLITS}; TAB annotator: {TAB_ANNOTATOR}; windows={WINDOW}/{OVERLAP}; seed={SEED}",
         f"- system check: median {STATS['median_latency_s']:.2f}s/window, {STATS['median_tokens_per_s']:.1f} tok/s, "
         f"valid JSON {STATS['valid_json_rate']:.2%}, decision agreement {STATS['decision_agreement']:.2%}, GPU share {GPU}",
         f"- windows={tot('n_windows')} truncated={tot('n_truncated')} invalid={tot('n_invalid')} unmatched strings={tot('n_unmatched')}", ""]
for scope in ["direct+quasi", "direct"]:
    lines += [f"## Metrics (TAB scope {scope}; CRAPII is scope-free)", "", "```", fmt_ci(TABLES[scope]).to_string(index=False), "```", ""]
lines += ["## Worst misses (short snippets)", "", "```", MISSES.head(10).to_string(index=False) if len(MISSES) else "none", "```", "",
          "## Worst false redactions of keep spans", "", "```", FALSE_RED.head(10).to_string(index=False) if len(FALSE_RED) else "none", "```"]
md_path = RESULTS_DIR / f"summary_{tag}.md"
md_path.write_text("\n".join(lines))
print("wrote:", md_path.relative_to(ROOT), csv_path.relative_to(ROOT), CHECK_PATH.relative_to(ROOT), sep="\n  ")